In [ ]:
import os
import glob
import math
import json
import torch
import pickle
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import tqdm
import numpy as np
from sklearn.cluster import KMeans
from argparse import ArgumentParser
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score
from matplotlib.patches import Rectangle

In [ ]:
### metrics
def singular_value_sum(S):
    return S.sum(-1)

def information_abundance(S):
    return S.sum(-1) / S.max(-1).values

def get_cov(data):
    m = data.mean(0, keepdim=True)
    rst = (data - m).T @ (data - m) / len(data)
    return rst

def uniformity(x):
    x = F.normalize(x, dim=-1)
    return torch.pdist(x, p=2).pow(2).mul(-2).exp().mean().log()

def RankMe(Z, epsilon=1.0e-7):
    # Z: [..., B, D]
    S = torch.svd(Z).S
    p_vector = S / S.sum(-1, keepdim=True) + epsilon
    rst = torch.exp((- p_vector * torch.log(p_vector)).sum(-1))
    return rst

def get_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
    return path

def count_files_in_folder(folder_path):
    file_count = 0
    for root, dirs, files in os.walk(folder_path):
        file_count += len(files)
    return file_count

In [ ]:
plot_dir = 'figure'

dataset_id_dict = {
    'Criteo': 'criteo_x1_7b681156',
    'Avazu': 'avazu_x4_3bbbc4c9',
}

dataset_name = 'Avazu'
#dataset_name = 'Criteo'
DATASET_ID = dataset_id_dict[dataset_name] # criteo_x1_7b681156, frappe_x1_04e961e9, avazu_x4_3bbbc4c9, criteo_x4_9ea3bdfc

############################## 

model_name_list = ['FM_avazu','DeepFM_avazu'] #Replace it with your expid.
embedding_dir = "" # Replace it with your file path.


##############################
printed_name = model_name_list
plot_tag = ''

early_stop_patience = 0

rst_dict = {}

num_field_dict = {
    'Criteo': 39, 'Avazu': 24,
}
num_field = num_field_dict[dataset_name]

def get_cardinality():
    path = os.path.join('data', dataset_name, DATASET_ID, 'feature_map.json')
    feature_map = json.load(open(path, 'r'))
    cardinality_list = []
    for _ in feature_map['features']:
        cardinality_list.append(list(_.values())[0].get('vocab_size', 0))
    return torch.tensor(cardinality_list)
cardinality = get_cardinality()
field_idx_sorted_by_cardinality = torch.sort(cardinality, descending=True).indices
num_field = len(cardinality)
num_interacted_field = num_field * (num_field - 1) // 2
row, col = torch.triu_indices(num_field, num_field, offset=1)
interacted_feature_cardinality = (cardinality.unsqueeze(-1) @ cardinality.unsqueeze(0))[row, col]
field_idx_sorted_by_interacted_feature_cardinality = torch.sort(interacted_feature_cardinality, descending=True).indices


log_data_split_dict = {
    'train': 0,
    'valid': 1,
    'test': 2,
}
phase = 'valid'
chunk_size = 5000
log_data_split = log_data_split_dict[phase]


def count_files_in_folder(folder_path):
    file_count = 0
    for root, dirs, files in os.walk(folder_path):
        file_count += len(files)
    return file_count
emb_list = []
grad_list = []
emb_list_all = []
batch_data = None 
batch_data_dir = os.path.join(f'{embedding_dir}/raw_data/{DATASET_ID}', "batch_data.pth") #Replace it with your file path.
for model_name in model_name_list:
    emb_dir = os.path.join(f'{embedding_dir}/raw_data/{DATASET_ID}/{model_name}/', "emb")
    file_count = count_files_in_folder(emb_dir)
    emb_list_all.append([])
    
    with open(os.path.join(emb_dir, f'emb_epoch0.pth'), 'rb') as handle:
        emb_dict = pickle.load(handle)
        print(emb_dict.keys())
        
        if 'emb_grad' not in emb_dict:
            emb_dict = {k: v.cpu() for k, v in emb_dict.items()}
        emb_list_all[-1].append(emb_dict)
        emb_list.append(emb_dict)

In [ ]:
fig, ax1 = plt.subplots()

label = ['FM','FM + parallel DNN', 'FM + stacked DNN']
color = ["#fbb463","#80b1d3","#bdbadb"]

sigular_value_list = []
for idx, emb in enumerate(emb_list):
    model_name = printed_name[idx]

    ### Feature Emb
    y = emb['feature_emb'].flatten(1)

    ### Gradient of Feature Emb
    # y = emb['emb_grad'][0].flatten(1)

    print(RankMe(y))
    y = get_cov(y)
    y = torch.svd(y).S
    sigular_value_list.append(y)
    y = y / y.max()
    y = y.log()
    
    x = list(range(len(y)))
    ax1.plot(x, y, label=label[idx], color=color[idx])
    ax1.scatter(x, y, s=1.5, color=color[idx])

ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.legend(fontsize=15)
ax1.set_xlabel('Dimension Index', fontsize=15)
ax1.set_ylabel('Log of Normalized Spectrum', fontsize=15)

with open('sigular_value_list.pkl', 'wb') as f:
    pickle.dump(sigular_value_list, f)

# plt.savefig('figure/grad_spectrum_nfm.png', dpi=300)
plt.show()